# P1 session A - validate before spending anything

Everything here is cheap. It exists so that a session B or C is never started on a corpus or a task that cannot support the paper.\n\nRun this first, read the last cell, and only continue if it says the gate passed.

**Settings: Accelerator `GPU T4 x2`, Internet `ON`.**

Nothing in this notebook configures the experiment. Every cell runs a script
from the repo; the design lives in `configs/experiment.yaml` and
`configs/p1_split_manifest.json`. If a check fails, stop and report it -- the
design is not adjusted to make a check pass.

In [ ]:
# 1. Get the code.
REPO_URL = "https://github.com/fairuz-anadi/quantization.git"
REF      = "main"          # pin to a commit SHA for the run that goes in the paper

import os, subprocess, sys
SRC = "/kaggle/working/quantlang"
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO_URL, SRC],
                   check=True)
print(subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())
os.chdir(SRC); sys.path.insert(0, SRC)

In [ ]:
# 2. Dependencies. Kaggle's torch is CUDA-matched -- never reinstall it.
!pip install -q -U "transformers>=4.45" "bitsandbytes>=0.43" "peft>=0.13" accelerate datasets pyyaml

In [ ]:
# 3. Environment probe. STOP HERE if this exits non-zero.
#    A P100 cannot run INT8 or NF4.
!python scripts/probe_env.py --outdir /kaggle/working

## The frozen contracts

`freeze_p0.py` proves P0 is byte-identical to what produced the published
results. `pytest` covers the P1 construction, including the checks that version
1 did not have: the substring shortcut is worth exactly a guess, the gold and
the distractors are present at the same rate, and the FT arm is verified to
differ from the Base arm.

In [ ]:
!python scripts/freeze_p0.py
!python -m pytest -q

## The corpus reproduces exactly

Re-derives every P1 item from the pinned dataset revision, the frozen
`split_seed` and the pinned tokenizer, and compares against the frozen digests.
Downloads the corpus, so it takes a few minutes.

In [ ]:
!python scripts/build_p1_splits.py --check

## The learnability gate

Two questions, cheapest first.

1. **CPU.** Does the frozen item set still carry a lexical shortcut? Version 1
   scored ~0.96 (English) and ~0.92 (Bangla) on "choose the option that appears
   verbatim in the passage", on a 100% gold-presence rate. Version 2 scores
   exactly 0.25 because all four options are present.
2. **GPU, a few minutes.** How well does the BASE model already do on the P1
   task, scored by the exact P0 evaluator? The stop threshold is P0's own best
   measured cell, read from `results/ALL_P0_RESULTS/tables/accuracy.csv` -- it
   is not a number chosen here. If the base model is above it, fine-tuning has
   no headroom and sessions B and C are not worth running.

In [ ]:
!python scripts/check_p1_learnability.py --langs eng_Latn ben_Beng --outdir /kaggle/working/p1_gate

## The 20-item smoke test

Nine checks. The ninth is new: it compares the fine-tuned logits against the
base model's. A full English run once passed checks 1-8 while producing logits
bit-identical to the base model at every precision, because nothing compared the
two arms.

In [ ]:
!python scripts/run_p1_smoke.py --outdir /kaggle/working/p1_smoke --lang eng_Latn

In [ ]:
# Read the report.
import json
r = json.load(open("/kaggle/working/p1_smoke/p1_smoke_report.json", encoding="utf-8"))
for name, check in sorted(r["checks"].items()):
    print(f"[{'PASS' if check['pass'] else 'FAIL'}] {name}")

d = r["checks"]["9_ft_arm_differs_from_base_arm"]
print("\nFT vs base, max logit delta:", d["max_logit_delta_vs_base"])
print("merge weight delta:", d["merge_weight_delta"])
print("\nALL CHECKS PASSED:", r["all_checks_passed"])

## Gate

Continue to session B only if:

* `freeze_p0.py` reports the P0 freeze intact (the 30 unregistered raw
  provenance files are a known pre-existing gap, not a failure);
* `pytest` is green;
* `--check` reports the split reproduces exactly;
* the learnability gate prints `GATE PASSED`;
* all nine smoke checks read PASS, and check 9's logit delta is **not** 0.0.

Then download `/kaggle/working/p1_gate/p1_learnability_report.json` and
`/kaggle/working/p1_smoke/p1_smoke_report.json`.

In [ ]:
import shutil, os
KEEP = "/kaggle/working/p1_sessionA_keep"
os.makedirs(KEEP, exist_ok=True)
for src in ["/kaggle/working/p1_gate/p1_learnability_report.json",
            "/kaggle/working/p1_smoke/p1_smoke_report.json"]:
    if os.path.exists(src):
        shutil.copy(src, KEEP)
if os.path.isdir("/kaggle/working/p1_smoke/adapter"):
    shutil.copytree("/kaggle/working/p1_smoke/adapter", f"{KEEP}/adapter",
                    dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/p1_sessionA", "zip", KEEP)

# The merged checkpoint is ~5.75 GB and is DERIVED -- rebuildable from the base
# model plus the adapter. Freeing it keeps the session inside the 20 GB limit.
if os.path.isdir("/kaggle/working/p1_smoke/merged"):
    shutil.rmtree("/kaggle/working/p1_smoke/merged")
print(sorted(os.listdir("/kaggle/working")))